# Photonic Wavelength Demultiplexer via Backpropagation

This notebook shows how gradient-based optimisation can train a photonic circuit to route different wavelengths to different detectors — exactly the way a neural network is trained, but the "weights" are physical waveguide parameters.

## The device: a wavelength demultiplexer

A wavelength-division multiplexing (WDM) demultiplexer is a chip-scale device that separates a multi-wavelength optical signal into individual channels. It is a workhorse of fibre-optic communications, data centre interconnects, and optical sensing.

We will design a **1→4 binary splitter tree** that routes four telecom wavelengths (1300–1330 nm) to four separate photodetectors. Light accumulates a different phase in each waveguide arm depending on the arm's effective refractive index `neff`. By tuning `neff`, we control how light interferes at each splitter output — a direct photonic analogue of adjusting neural network weights.

## Why backprop wins

The demux has **6 tunable phase parameters**. A finite-difference approach would need 7 forward simulations per gradient step (one baseline + one perturbed per parameter). Backpropagation needs only **1 forward + 1 backward** — a fixed cost regardless of how many parameters there are.

Because circulax is built on JAX, `jax.grad` gives us exact gradients through the full circuit solve at essentially zero overhead over the forward pass.

| Method | Evaluations per step | Scales with N? |
|--------|---------------------|----------------|
| Finite differences | N + 1 | Yes — O(N) |
| Backpropagation (circulax) | ~2 | No — O(1) |

## Key physics

At wavelength λ, a waveguide of length L and effective index `neff` accumulates phase:

$$\phi = \frac{2\pi \cdot n_{\text{eff}} \cdot L}{\lambda}$$

When two arms of an interferometer have different phases, they interfere constructively or destructively at the combiner outputs. By choosing `neff` values such that arm A constructively interferes at λ₁ and destructively at λ₂, we achieve wavelength selectivity.

In [ ]:
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np
import optax

from circulax.compiler import compile_netlist
from circulax.solvers import analyze_circuit
from circulax.utils import update_group_params, update_params_dict
from circulax.components.electronic import Resistor
from circulax.components.photonic import OpticalSource, OpticalWaveguide, Splitter

# 64-bit precision is important for photonic circuits: small changes in neff
# produce small changes in phase, and gradients can be tiny in early training.
jax.config.update("jax_enable_x64", True)

plt.rcParams.update({
    "figure.figsize": (8, 3.5),
    "axes.grid": True,
    "lines.color": "grey",
    "patch.edgecolor": "grey",
    "text.color": "grey",
    "axes.facecolor": "white",
    "axes.edgecolor": "grey",
    "axes.labelcolor": "grey",
    "xtick.color": "grey",
    "ytick.color": "grey",
    "grid.color": "grey",
    "figure.facecolor": "white",
    "figure.edgecolor": "white",
    "savefig.facecolor": "white",
    "savefig.edgecolor": "white",
})

## Circuit topology: binary splitter tree

```
Laser ── WG_in ── Spl_1 ──── WG_A ──── Spl_2 ──── WG_A1 ──── Det_1
                   │                     │
                   │                     └── WG_A2 ──── Det_2
                   │
                   └── WG_B ──── Spl_3 ──── WG_B1 ──── Det_3
                                   │
                                   └── WG_B2 ──── Det_4
```

**Fixed components:**
- `Laser` — ideal CW optical source at 1 W
- `WG_in` — 50 µm input coupling waveguide, neff = 2.4 (fixed)
- `Spl_1`, `Spl_2`, `Spl_3` — 50/50 Y-junction splitters
- `Det_1`–`Det_4` — photodetectors modelled as 1 Ω resistors

**Trainable parameters (6 total):**
- `neff` of `WG_A`, `WG_B` — first-level phase arms (control λ₁+λ₂ vs λ₃+λ₄ routing)
- `neff` of `WG_A1`, `WG_A2` — second-level arms in upper branch
- `neff` of `WG_B1`, `WG_B2` — second-level arms in lower branch

All phase-delay waveguides are 200 µm long. We slightly perturb initial neff values to break symmetry and give the optimiser a starting gradient.

**Target routing:**
| Wavelength | Target detector |
|-----------|----------------|
| λ₁ = 1300 nm | Det_1 |
| λ₂ = 1310 nm | Det_2 |
| λ₃ = 1320 nm | Det_3 |
| λ₄ = 1330 nm | Det_4 |

In [ ]:
# ── Component model registry ───────────────────────────────────────────────────
models = {
    "ground":    lambda: 0,
    "source":    OpticalSource,
    "waveguide": OpticalWaveguide,
    "splitter":  Splitter,
    "resistor":  Resistor,
}

# ── SAX-format netlist ─────────────────────────────────────────────────────────
#
# The netlist describes topology only — component values can be updated later
# without recompiling the circuit.
#
# neff values are deliberately slightly different to break the perfect symmetry
# that would give every detector equal power and zero gradient.
net_dict = {
    "instances": {
        "GND":   {"component": "ground"},
        "Laser": {"component": "source",    "settings": {"power": 1.0, "phase": 0.0}},
        # Fixed input coupling arm
        "WG_in": {"component": "waveguide", "settings": {"length_um": 50.0,  "neff": 2.4,  "loss_dB_cm": 0.1, "center_wavelength_nm": 1310.0}},
        # Splitters
        "Spl_1": {"component": "splitter",  "settings": {"split_ratio": 0.5}},
        "Spl_2": {"component": "splitter",  "settings": {"split_ratio": 0.5}},
        "Spl_3": {"component": "splitter",  "settings": {"split_ratio": 0.5}},
        # First-level phase arms (WG_A routes to Det_1+Det_2; WG_B to Det_3+Det_4)
        "WG_A":  {"component": "waveguide", "settings": {"length_um": 200.0, "neff": 2.40, "loss_dB_cm": 0.1, "center_wavelength_nm": 1310.0}},
        "WG_B":  {"component": "waveguide", "settings": {"length_um": 200.0, "neff": 2.41, "loss_dB_cm": 0.1, "center_wavelength_nm": 1310.0}},
        # Second-level phase arms
        "WG_A1": {"component": "waveguide", "settings": {"length_um": 200.0, "neff": 2.40, "loss_dB_cm": 0.1, "center_wavelength_nm": 1310.0}},
        "WG_A2": {"component": "waveguide", "settings": {"length_um": 200.0, "neff": 2.42, "loss_dB_cm": 0.1, "center_wavelength_nm": 1310.0}},
        "WG_B1": {"component": "waveguide", "settings": {"length_um": 200.0, "neff": 2.40, "loss_dB_cm": 0.1, "center_wavelength_nm": 1310.0}},
        "WG_B2": {"component": "waveguide", "settings": {"length_um": 200.0, "neff": 2.43, "loss_dB_cm": 0.1, "center_wavelength_nm": 1310.0}},
        # Photodetectors (1 Ω resistors provide a matched load)
        "Det_1": {"component": "resistor",  "settings": {"R": 1.0}},
        "Det_2": {"component": "resistor",  "settings": {"R": 1.0}},
        "Det_3": {"component": "resistor",  "settings": {"R": 1.0}},
        "Det_4": {"component": "resistor",  "settings": {"R": 1.0}},
    },
    "connections": {
        # Ground: laser return + all detector returns
        "GND,p1":   ("Laser,p2", "Det_1,p2", "Det_2,p2", "Det_3,p2", "Det_4,p2"),
        # Laser → input waveguide → first splitter
        "Laser,p1": "WG_in,p1",
        "WG_in,p2": "Spl_1,p1",
        # First split: upper branch (A) and lower branch (B)
        "Spl_1,p2": "WG_A,p1",
        "Spl_1,p3": "WG_B,p1",
        # Upper branch to second splitter
        "WG_A,p2":  "Spl_2,p1",
        # Lower branch to third splitter
        "WG_B,p2":  "Spl_3,p1",
        # Upper second-level arms to detectors
        "Spl_2,p2": "WG_A1,p1",
        "Spl_2,p3": "WG_A2,p1",
        "WG_A1,p2": "Det_1,p1",
        "WG_A2,p2": "Det_2,p1",
        # Lower second-level arms to detectors
        "Spl_3,p2": "WG_B1,p1",
        "Spl_3,p3": "WG_B2,p1",
        "WG_B1,p2": "Det_3,p1",
        "WG_B2,p2": "Det_4,p1",
    },
}

# ── Compile the netlist ────────────────────────────────────────────────────────
#
# compile_netlist runs once and returns:
#   groups   — dict of ComponentGroup objects (vectorised, JAX-traceable)
#   sys_size — number of scalar unknowns in y (real nodes + state variables)
#   port_map — dict mapping "Instance,port" -> index in y
#
# For photonic circuits the solver unfolds complex fields into a real
# 2·sys_size vector: y[0:sys_size] = Re(E), y[sys_size:] = Im(E).
# is_complex=True tells analyze_circuit to build the 2N×2N Jacobian.
groups, sys_size, port_map = compile_netlist(net_dict, models)

print(f"System size: {sys_size} real nodes")
print(f"Complex system size (2N): {sys_size * 2}")
print(f"Component groups: {list(groups.keys())}")
print(f"Detector node indices: " +
      ", ".join(f"Det_{i+1}={port_map[f'Det_{i+1},p1']}" for i in range(4)))

# Build the solver strategy once — this pre-computes the Jacobian sparsity
# pattern and allocates the KLU/dense factorisation structure.
# Because we call this outside of vmap/grad, it can do Python-level work.
solver = analyze_circuit(groups, sys_size, backend="dense", is_complex=True)
print("\nSolver ready.")

## Helper: compute the 4×4 power routing matrix

For each of the 4 wavelengths we solve the circuit and read the power absorbed at each detector. The result is a 4×4 matrix:

```
power[i, j] = fraction of total power at Det_{j+1} when driving at wavelength[i]
```

An ideal demux has power[i, i] ≈ 1 and all off-diagonal entries ≈ 0 (a perfect identity matrix).

### Implementation details

For photonic circuits, circulax unfolds complex fields `E = Re(E) + j·Im(E)` into a real vector of length `2·sys_size`:

```
y_flat[k]            = Re(E_k)   for k in 0..sys_size-1
y_flat[k + sys_size] = Im(E_k)   for k in 0..sys_size-1
```

So the complex field at detector node `n` is:
```python
E = y_flat[n] + 1j * y_flat[n + sys_size]
power = |E|²
```

We use `jax.vmap` to evaluate all 4 wavelengths in a single parallel call.

In [ ]:
# Target wavelengths (nm)
TARGET_WLS = jnp.array([1300.0, 1310.0, 1320.0, 1330.0])

# Detector port node indices in y
det_nodes = jnp.array([port_map[f"Det_{i+1},p1"] for i in range(4)])

# Initial guess for the complex DC solve:
# ones rather than zeros avoids the degenerate Newton starting point
# that can stall convergence in photonic circuits.
Y_GUESS = jnp.ones(sys_size * 2)


def get_power_matrix(neff_params, wavelengths=TARGET_WLS):
    """Compute the 4×4 power routing matrix for given neff values.

    Args:
        neff_params: Array of shape (6,) — [neff_A, neff_B, neff_A1, neff_A2, neff_B1, neff_B2]
        wavelengths: Array of wavelengths in nm, shape (4,). Defaults to TARGET_WLS.

    Returns:
        power: Array of shape (4, 4), normalised so each row sums to ~1.
                power[i, j] = fraction of power reaching Det_{j+1} at wavelengths[i].
    """
    neff_A, neff_B, neff_A1, neff_A2, neff_B1, neff_B2 = neff_params

    # Functionally update each phase arm's neff.
    # update_params_dict uses eqx.tree_at — no in-place mutation, fully
    # JAX-traceable and compatible with jax.grad.
    grps = update_params_dict(groups,  "waveguide", "WG_A",  "neff", neff_A)
    grps = update_params_dict(grps,    "waveguide", "WG_B",  "neff", neff_B)
    grps = update_params_dict(grps,    "waveguide", "WG_A1", "neff", neff_A1)
    grps = update_params_dict(grps,    "waveguide", "WG_A2", "neff", neff_A2)
    grps = update_params_dict(grps,    "waveguide", "WG_B1", "neff", neff_B1)
    grps = update_params_dict(grps,    "waveguide", "WG_B2", "neff", neff_B2)

    def solve_at_wavelength(wl):
        """Solve for one wavelength and return power at each detector."""
        # Update wavelength_nm for ALL waveguide instances at once.
        # update_group_params broadcasts the new value to every element in the group.
        grps_wl = update_group_params(grps, "waveguide", "wavelength_nm", wl)

        # solve_dc runs Newton-Raphson inside JAX — compatible with vmap and grad.
        # The solver object was created outside this function, so it is a static
        # captured closure (not a traced argument).
        y_flat = solver.solve_dc(grps_wl, Y_GUESS)

        # Reconstruct complex field at each detector port from the unrolled vector.
        # y_flat = [Re(E_0), ..., Re(E_{N-1}), Im(E_0), ..., Im(E_{N-1})]
        E_det = y_flat[det_nodes] + 1j * y_flat[det_nodes + sys_size]
        return jnp.abs(E_det) ** 2

    # vmap evaluates solve_at_wavelength for every wavelength in parallel.
    # JAX traces the function once and vectorises it — no Python loop.
    powers = jax.vmap(solve_at_wavelength)(wavelengths)  # shape: (4, 4)

    # Normalise each row by total power so values represent routing fractions.
    row_totals = jnp.sum(powers, axis=1, keepdims=True) + 1e-12
    return powers / row_totals


# Initial neff values — slight asymmetry to break symmetry degeneracy
neff_init = jnp.array([2.40, 2.41, 2.40, 2.42, 2.40, 2.43])

print("Computing initial power routing matrix (JIT compiling + solving)...")
power_init = jax.jit(get_power_matrix)(neff_init)
print("Done.")
print("\nInitial 4×4 power routing matrix:")
print("Rows = wavelengths [1300, 1310, 1320, 1330 nm]")
print("Cols = detectors [Det_1, Det_2, Det_3, Det_4]")
print(np.array(power_init).round(3))

## Part 1 — Baseline: routing before optimisation

With nearly identical neff values, all arms accumulate similar phase and the splitter tree distributes light roughly equally to all four detectors. The power matrix should be close to a flat 0.25 everywhere — no wavelength selectivity.

In [ ]:
def plot_power_matrix(power_matrix, title, ax=None):
    """Plot a 4×4 power routing matrix as an annotated heatmap."""
    if ax is None:
        fig, ax = plt.subplots(figsize=(5, 4))
    else:
        fig = ax.figure

    pm = np.array(power_matrix)
    im = ax.imshow(pm, vmin=0, vmax=1, cmap="viridis", aspect="auto")
    fig.colorbar(im, ax=ax, label="Fraction of power")

    ax.set_xticks(range(4))
    ax.set_yticks(range(4))
    ax.set_xticklabels(["Det_1", "Det_2", "Det_3", "Det_4"], color="grey")
    ax.set_yticklabels(["1300 nm", "1310 nm", "1320 nm", "1330 nm"], color="grey")
    ax.set_xlabel("Detector", color="grey")
    ax.set_ylabel("Wavelength", color="grey")
    ax.set_title(title, color="grey")

    # Annotate each cell with the percentage value
    for i in range(4):
        for j in range(4):
            val = pm[i, j]
            text_color = "white" if val < 0.5 else "black"
            ax.text(j, i, f"{val*100:.1f}%",
                    ha="center", va="center", fontsize=9, color=text_color)

    return fig, ax


fig, ax = plt.subplots(figsize=(5, 4))
plot_power_matrix(power_init, "Power Routing Matrix — Before Training", ax=ax)
plt.tight_layout()
plt.show()

# Compute and display the "contrast" score: diagonal / row total
# A perfect demux has contrast = 1.0 for every wavelength.
diag_powers = np.diag(np.array(power_init))
print(f"\nInitial routing contrast (correct detector fraction):")
for i, (wl, c) in enumerate(zip([1300, 1310, 1320, 1330], diag_powers)):
    print(f"  λ={wl} nm → Det_{i+1}: {c*100:.1f}%")
print(f"  Mean contrast: {diag_powers.mean()*100:.1f}% (ideal = 100%)")

## Part 2 — Gradient-based optimisation

### Loss function

We maximise the **routing contrast**: the fraction of each wavelength's total power that reaches its designated detector.

$$\mathcal{L} = -\frac{1}{4} \sum_{i=1}^{4} \frac{P_{ii}}{\sum_j P_{ij}}$$

This is `−mean(diagonal of normalised power matrix)`. We minimise the loss, so minimising `−contrast` = maximising contrast.

### Why this loss?

The normalised power matrix rows already sum to 1.0, so `power[i, i]` is exactly the fraction of wavelength `i`'s power that reaches its target. A perfect demux would give `loss = −1.0`. A random router gives `loss ≈ −0.25`.

In [ ]:
def loss_fn(neff_params):
    """Negative mean routing contrast — minimise to maximise wavelength selectivity.

    Returns a scalar in [-1, 0]:
      -1.0 = perfect demux (all power at correct detector)
       0.0 = worst case (no power at any correct detector)
    """
    # power shape: (4, 4), already row-normalised
    power = get_power_matrix(neff_params)

    # Diagonal entries: power at the "correct" detector for each wavelength
    correct_fractions = jnp.diag(power)

    # Minimise negative contrast (= maximise contrast)
    return -jnp.mean(correct_fractions)


# Verify the loss is differentiable: compute loss AND gradient in one pass.
# jax.value_and_grad is strictly more efficient than calling loss_fn and
# jax.grad(loss_fn) separately — it reuses the forward-pass computation.
print("Computing loss and gradients at initial parameters...")
loss_val, grads = jax.jit(jax.value_and_grad(loss_fn))(neff_init)
print(f"  Initial loss:    {float(loss_val):.4f}  (random ≈ -0.25, perfect = -1.0)")
print(f"  Mean contrast:   {-float(loss_val)*100:.1f}%")
print(f"  Gradient norms:  {[f'{float(g):.4f}' for g in grads]}")
print()
print("Gradient labels: [neff_A, neff_B, neff_A1, neff_A2, neff_B1, neff_B2]")
print("Non-zero gradients confirm the circuit is differentiable end-to-end.")

In [ ]:
# ── Adam optimisation loop ─────────────────────────────────────────────────────
#
# We train for 500 steps. Adam adapts the learning rate per-parameter,
# which helps because the neff gradient magnitudes vary across arms.
#
# The learning rate 5e-4 is conservative — neff changes of ~1e-3 produce
# phase shifts of ~1 radian over 200 µm at 1310 nm, so we want small steps.

N_STEPS = 500
LR      = 5e-4

optimizer  = optax.adam(learning_rate=LR)
neff_params = neff_init
opt_state   = optimizer.init(neff_params)

# Pre-compile the gradient function once
value_and_grad_fn = jax.jit(jax.value_and_grad(loss_fn))

losses          = []
neff_history    = []

print(f"Training for {N_STEPS} steps (Adam, lr={LR})...")
print(f"{'Step':>5}  {'Loss':>8}  {'Contrast':>10}  {'neff_A':>8}  {'neff_B':>8}")
print("-" * 52)

for step in range(N_STEPS):
    loss, grads = value_and_grad_fn(neff_params)
    losses.append(float(loss))
    neff_history.append(np.array(neff_params))

    updates, opt_state = optimizer.update(grads, opt_state)
    neff_params = optax.apply_updates(neff_params, updates)

    if step == 0 or (step + 1) % 100 == 0:
        print(f"{step+1:5d}  {float(loss):8.4f}  {-float(loss)*100:9.1f}%  "
              f"{float(neff_params[0]):8.5f}  {float(neff_params[1]):8.5f}")

neff_history = np.array(neff_history)
print(f"\nFinal neff parameters:")
labels = ["neff_A", "neff_B", "neff_A1", "neff_A2", "neff_B1", "neff_B2"]
for label, p_init, p_final in zip(labels, neff_init, neff_params):
    delta = float(p_final) - float(p_init)
    print(f"  {label}: {float(p_init):.5f} → {float(p_final):.5f}  (Δ = {delta:+.5f})")

In [ ]:
# ── Plot: optimisation convergence ─────────────────────────────────────────────

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.5))

# Loss curve (inverted so we see contrast climbing toward 100%)
contrast_history = [-l * 100 for l in losses]
ax1.plot(contrast_history, color="C0", lw=2)
ax1.axhline(25.0, color="grey", ls=":", lw=1, label="Random baseline (25%)")
ax1.axhline(100.0, color="C1", ls="--", lw=1, label="Perfect demux (100%)")
ax1.set_xlabel("Optimisation step")
ax1.set_ylabel("Mean routing contrast (%)")
ax1.set_title("Convergence")
ax1.set_ylim(0, 105)
ax1.legend(fontsize=8)

# neff parameter trajectories
colors = ["C0", "C1", "C2", "C3", "C4", "C5"]
for i, (label, color) in enumerate(zip(labels, colors)):
    ax2.plot(neff_history[:, i], color=color, lw=1.5, label=label)
ax2.set_xlabel("Optimisation step")
ax2.set_ylabel("neff")
ax2.set_title("Phase parameter trajectories")
ax2.legend(fontsize=8, ncol=2)

plt.tight_layout()
plt.show()

In [ ]:
# ── Final power routing matrix ─────────────────────────────────────────────────

print("Computing final power routing matrix...")
power_final = jax.jit(get_power_matrix)(neff_params)

fig, ax = plt.subplots(figsize=(5, 4))
plot_power_matrix(power_final, "Power Routing Matrix — After Training", ax=ax)
plt.tight_layout()
plt.show()

diag_final = np.diag(np.array(power_final))
print(f"\nFinal routing contrast:")
for i, (wl, c) in enumerate(zip([1300, 1310, 1320, 1330], diag_final)):
    bar = "█" * int(c * 30)
    print(f"  λ={wl} nm → Det_{i+1}: {c*100:5.1f}%  {bar}")
print(f"  Mean contrast: {diag_final.mean()*100:.1f}%")

In [ ]:
# ── Side-by-side comparison: before vs after ───────────────────────────────────

fig, (ax_before, ax_after) = plt.subplots(1, 2, figsize=(11, 4))
plot_power_matrix(power_init,  "Before Training", ax=ax_before)
plot_power_matrix(power_final, "After Training",  ax=ax_after)
fig.suptitle("Photonic Demultiplexer — Power Routing Matrix", color="grey", fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# ── Wavelength sweep: transmission spectra at each detector ────────────────────
#
# Sweep a dense set of wavelengths to reveal each detector's passband shape.
# A good demux shows four non-overlapping peaks, one per detector.

sweep_wls = jnp.linspace(1280.0, 1350.0, 200)

# We need unnormalised power here (not normalised by row total)
# to show the actual passband shape including insertion loss.
def get_raw_power_sweep(neff_params, wavelengths):
    """4×N power matrix for a wavelength sweep (not row-normalised)."""
    neff_A, neff_B, neff_A1, neff_A2, neff_B1, neff_B2 = neff_params
    grps = update_params_dict(groups,  "waveguide", "WG_A",  "neff", neff_A)
    grps = update_params_dict(grps,    "waveguide", "WG_B",  "neff", neff_B)
    grps = update_params_dict(grps,    "waveguide", "WG_A1", "neff", neff_A1)
    grps = update_params_dict(grps,    "waveguide", "WG_A2", "neff", neff_A2)
    grps = update_params_dict(grps,    "waveguide", "WG_B1", "neff", neff_B1)
    grps = update_params_dict(grps,    "waveguide", "WG_B2", "neff", neff_B2)

    def solve_wl(wl):
        grps_wl = update_group_params(grps, "waveguide", "wavelength_nm", wl)
        y_flat  = solver.solve_dc(grps_wl, Y_GUESS)
        E_det   = y_flat[det_nodes] + 1j * y_flat[det_nodes + sys_size]
        return jnp.abs(E_det) ** 2

    return jax.vmap(solve_wl)(wavelengths)  # shape: (N_wl, 4)


print("Running wavelength sweep (vmap over 200 wavelengths)...")
sweep_power = jax.jit(get_raw_power_sweep)(neff_params, sweep_wls)
print("Done.")

fig, ax = plt.subplots(figsize=(9, 4))
sweep_wls_np = np.array(sweep_wls)
sweep_power_np = np.array(sweep_power)  # shape: (200, 4)

target_wls_nm = [1300, 1310, 1320, 1330]
det_colors = ["C0", "C1", "C2", "C3"]

for j in range(4):
    ax.plot(sweep_wls_np, sweep_power_np[:, j],
            color=det_colors[j], lw=2, label=f"Det_{j+1} (target: {target_wls_nm[j]} nm)")
    # Mark target wavelength
    ax.axvline(target_wls_nm[j], color=det_colors[j], ls=":", lw=1, alpha=0.6)

ax.set_xlabel("Wavelength (nm)")
ax.set_ylabel("Received power (W)")
ax.set_title("Transmission spectra after training — each detector peaks at its target wavelength")
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

## Part 3 — Why backpropagation wins: the scaling argument

The demux we just trained has **N = 6** tunable parameters. But modern photonic integrated circuits can have hundreds of tunable elements — microring resonators, phase shifters, Mach-Zehnder meshes. How does the cost of optimisation scale?

### Finite-difference gradient estimation

To estimate `∂L/∂neff_k` with a forward-difference scheme:

$$\frac{\partial \mathcal{L}}{\partial n_k} \approx \frac{\mathcal{L}(n + \epsilon e_k) - \mathcal{L}(n)}{\epsilon}$$

you need **1 baseline + N perturbations = N + 1 circuit solves per gradient step**.

### Backpropagation (automatic differentiation)

`jax.grad` computes the **exact** gradient for all N parameters simultaneously in roughly **1 forward + 1 backward pass ≈ 2 circuit solves**, regardless of N. The backward pass is the vector-Jacobian product — it costs the same as one additional forward sweep.

### Practical speedup

For our 6-parameter circuit:
- FD needs 7 evaluations per step
- Backprop needs ~2 evaluations per step
- Speedup: **3.5×** even at this small scale

For a 100-element photonic mesh, the speedup is **50×**. FD becomes impractical; backprop remains cheap.

In [ ]:
# ── Scaling comparison: FD vs backprop ────────────────────────────────────────

N_values    = [6, 10, 20, 50, 100, 500]
fd_evals    = [N + 1 for N in N_values]           # central difference would be 2N+1
bp_evals    = [2] * len(N_values)                  # forward + backward, always 2
speedup     = [fd / bp for fd, bp in zip(fd_evals, bp_evals)]

print(f"{'N params':>10} | {'FD evals':>10} | {'Backprop evals':>15} | {'Speedup':>8}")
print("-" * 52)
for N, fd, bp, sp in zip(N_values, fd_evals, bp_evals, speedup):
    print(f"{N:>10} | {fd:>10} | {bp:>15} | {sp:>7.1f}×")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.5))

# Left: evaluations per gradient step
ax1.plot(N_values, fd_evals, "o-", color="C3", lw=2, ms=6, label="Finite differences")
ax1.plot(N_values, bp_evals, "s-", color="C0", lw=2, ms=6, label="Backprop (circulax)")
ax1.set_xlabel("Number of tunable parameters")
ax1.set_ylabel("Circuit evaluations per gradient step")
ax1.set_title("Cost scaling")
ax1.legend(fontsize=9)

# Right: speedup
ax2.plot(N_values, speedup, "D-", color="C2", lw=2, ms=6)
ax2.axvline(6, color="grey", ls=":", lw=1)
ax2.annotate("This demo (N=6)\n3.5× speedup",
             xy=(6, 3.5), xytext=(20, 4.5),
             fontsize=8, color="grey",
             arrowprops=dict(arrowstyle="->", color="grey", lw=0.8))
ax2.set_xlabel("Number of tunable parameters")
ax2.set_ylabel("Speedup over finite differences")
ax2.set_title("Backprop speedup vs finite differences")

plt.tight_layout()
plt.show()

## Summary

We trained a photonic wavelength demultiplexer using gradient descent, starting from near-uniform neff values (no selectivity) and converging to a design where each of the four target wavelengths is preferentially routed to its designated detector.

### What made this possible?

| Step | Tool | Role |
|------|------|------|
| Circuit compilation | `compile_netlist` | Runs once; produces JAX-traceable `ComponentGroup` objects |
| Parameter updates | `update_params_dict` | `eqx.tree_at` functional update; no re-compilation, fully differentiable |
| Wavelength sweep | `jax.vmap` | Evaluates all wavelengths in one parallel call; vectorised at the XLA level |
| Exact gradients | `jax.grad` | Reverse-mode AD through Newton-Raphson solve and Y-matrix assembly |
| Optimisation | `optax.adam` | Standard first-order optimiser; one gradient step ≈ 2 circuit solves |

### Going further

The same pattern extends directly to:

- **Larger photonic meshes** — Mach-Zehnder interferometer arrays, microring add-drop multiplexers, photonic neural network layers. The optimisation cost stays constant at ~2 solves per step.
- **Multi-objective design** — add insertion loss, polarisation crosstalk, or fabrication tolerance penalties to the loss function with no API changes.
- **Inverse design** — if you can measure a real device's S-parameters and compare them to the simulator output, `jax.grad` gives you the fabrication error gradient for free.
- **Electronic circuits** — swap the photonic components for Resistors, MOSFETs, or Op-Amps; the optimisation loop is identical.